# Cyclone Mocha — DTM Analysis & Sentinel Asia VAPs (May 2023)

Cyclone Mocha made landfall near Sittwe, Rakhine State on **14 May 2023** — one of the
most intense storms ever recorded in the Bay of Bengal, also causing severe flooding
along the Bangladesh/Myanmar border.

This notebook combines two analyses of the event:

**Part A — Digital Terrain Model (Rakhine State, Myanmar)**
- Copernicus GLO-30 vs GLO-90 elevation comparison for the landfall zone
- Ocean masking (elevation ≤ 0 m → NaN)

**Part B — Sentinel Asia SAR Flood-Extent VAPs (Bangladesh / Chittagong coast)**

| Layer | Sensor | Date | Producer |
|---|---|---|---|
| JAXA Flood Proxy Map (automatic) | ALOS-2 SAR | 15 May 2023 | JAXA |
| Detected Water — Chittagong Province | ALOS-2 SAR | 15 May 2023 | AIT |
| Flooded Areas — Bangladesh | Sentinel-1 SAR | 18 May 2023 | MBRSC |

DEM data is accessed cloud-natively via the
[Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/) STAC API.
VAP layers are tiled PNG map services from
[Sentinel Asia](https://sentinel-asia.org/EO/2023/article20230514BD.html),
served as ArcGIS XYZ/WMTS tile services.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import xarray as xr
import rioxarray as rxr
from rioxarray.merge import merge_arrays
import pystac_client
import planetary_computer
import hvplot.xarray

import holoviews as hv
import geoviews as gv
import geoviews.tile_sources as gvts
from holoviews import opts
import panel as pn

hv.extension('bokeh')
gv.extension('bokeh')
pn.extension()

# --- Part A: Rakhine State / landfall zone (DEM) ---
BBOX = [90.9, 17.5, 94.6, 21.3]   # [lon_min, lat_min, lon_max, lat_max]

# --- Part B: Bangladesh coast / Chittagong region (VAP layers, Web Mercator) ---
XLIM = (9_900_000, 10_500_000)   # ~lon 88.9 – 94.3
YLIM = (2_050_000, 2_650_000)    # ~lat 18.2 – 23.2

TILE_BASE = (
    "https://tiles.arcgis.com/tiles/Ce7cHX9rBcQKrxER"
    "/arcgis/rest/services/{name}/MapServer/tile/{Z}/{Y}/{X}"
)

SERVICES = {
    "JAXA — ALOS-2 (15 May)": "JAXA_by_ALOS2_20230515",
    "AIT — ALOS-2 (15 May)": "AIT_by_ALOS2_20230515",
    "MBRSC — Sentinel-1 (18 May)": "MBRSC_by_Sentinel1_20230518",
}

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Setup complete")

---
## Part A — Digital Terrain Model (Rakhine State)

## 2. DTM — Copernicus GLO-30 vs GLO-90 comparison

The Copernicus Digital Elevation Model is available at two resolutions on Planetary Computer:
- **`cop-dem-glo-30`** — 30 m posting (~1 arc-second)
- **`cop-dem-glo-90`** — 90 m posting (~3 arc-second)

Loading both lets us compare the terrain detail trade-off.

In [ ]:
items_30 = catalog.search(collections=["cop-dem-glo-30"], bbox=BBOX).item_collection()
items_90 = catalog.search(collections=["cop-dem-glo-90"], bbox=BBOX).item_collection()
print(f"GLO-30 items: {len(items_30)}")
print(f"GLO-90 items: {len(items_90)}")

In [ ]:
def load_dem(items):
    """Open each DEM tile individually, merge them, then clip to BBOX."""
    tiles = []
    for item in items:
        href = item.assets["data"].href
        da = rxr.open_rasterio(href, chunks="auto").squeeze("band", drop=True)
        tiles.append(da)
    merged = merge_arrays(tiles)
    clipped = merged.rio.clip_box(*BBOX)
    return clipped.compute()

print("Loading GLO-30 tiles...")
dem_30 = load_dem(items_30)
print(f"GLO-30 shape: {dem_30.shape},  min={float(dem_30.min()):.1f} m,  max={float(dem_30.max()):.1f} m,  mean={float(dem_30.mean()):.1f} m")

print("Loading GLO-90 tiles...")
dem_90 = load_dem(items_90)
print(f"GLO-90 shape: {dem_90.shape},  min={float(dem_90.min()):.1f} m,  max={float(dem_90.max()):.1f} m,  mean={float(dem_90.mean()):.1f} m")

In [ ]:
crs_dem = dem_30.rio.crs.to_string() if dem_30.rio.crs else "EPSG:4326"

p30 = dem_30.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-30 (30 m)",
)

p90 = dem_90.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-90 (90 m)",
)

p30 + p90

## 3. Ocean Masking

The Bay of Bengal lies to the **west** of the study area.  
Copernicus DEM encodes ocean/sea pixels as zero or negative values.  
We mask all pixels with elevation **≤ 0 m** to `NaN`, producing land-only DEMs
(`dem_30_land` and `dem_90_land`) that can be used without the ocean dominating
colour scales or statistics.

In [ ]:
# Mask ocean: elevation <= 0 → NaN
dem_30_land = dem_30.where(dem_30 > 0)
dem_90_land = dem_90.where(dem_90 > 0)

print(f"GLO-30 land  —  min={float(dem_30_land.min()):.1f} m,  "
      f"max={float(dem_30_land.max()):.1f} m,  "
      f"mean={float(dem_30_land.mean()):.1f} m")
print(f"GLO-90 land  —  min={float(dem_90_land.min()):.1f} m,  "
      f"max={float(dem_90_land.max()):.1f} m,  "
      f"mean={float(dem_90_land.mean()):.1f} m")

In [ ]:
p30_land = dem_30_land.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-30 — land only (ocean = NaN)",
)

p90_land = dem_90_land.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-90 — land only (ocean = NaN)",
)

p30_land + p90_land

---
## Part B — Sentinel Asia SAR Flood-Extent VAPs (Bangladesh / Chittagong)

## 4. Individual VAP Layers

Each of the three SAR-derived flood-extent products is loaded as a GeoViews `WMTS`
tile source and overlaid on an Esri satellite basemap.

In [ ]:
def vap_tile(service_name):
    url = TILE_BASE.format(name=service_name, Z="{Z}", Y="{Y}", X="{X}")
    return gv.WMTS(url)

basemap = gvts.EsriImagery

shared_opts = opts.WMTS(
    width=500, height=450,
    xlim=XLIM, ylim=YLIM,
)

plots = []
for title, svc in SERVICES.items():
    layer = basemap * vap_tile(svc)
    layer = layer.opts(
        shared_opts,
        opts.Overlay(title=title),
    )
    plots.append(layer)

hv.Layout(plots).cols(2)

## 5. Combined Overlay — All Three VAPs

Stack all three flood layers on a single satellite basemap to compare spatial coverage.

In [ ]:
combined = basemap
for svc in SERVICES.values():
    combined = combined * vap_tile(svc)

combined.opts(
    opts.WMTS(width=750, height=650, xlim=XLIM, ylim=YLIM),
    opts.Overlay(title="All VAPs — Cyclone Mocha flood extent (May 2023)"),
)

## 6. Interactive Selector — Choose Layer with a Widget

In [ ]:
layer_select = pn.widgets.Select(
    name="VAP layer",
    options=list(SERVICES.keys()),
    value=list(SERVICES.keys())[0],
)

@pn.depends(layer_select)
def show_layer(title):
    svc = SERVICES[title]
    return (
        basemap * vap_tile(svc)
    ).opts(
        opts.WMTS(width=700, height=600, xlim=XLIM, ylim=YLIM),
        opts.Overlay(title=title),
    )

pn.Column(layer_select, show_layer)

---
## Part C — Cyclone Mocha Track from Copernicus ERA5

The cyclone track is derived entirely from **ERA5 reanalysis** — the global atmospheric reanalysis produced by ECMWF and distributed through the [Copernicus Climate Data Store (CDS)](https://cds.climate.copernicus.eu/).

**Method:** for each 3-hourly timestep over 8–15 May 2023, the grid point of **minimum mean sea-level pressure (MSL)** within the Bay of Bengal domain is taken as the cyclone centre. Track intensity is expressed as the 10 m wind speed at that centre point.

> **Prerequisite:** a free [Copernicus CDS account](https://cds.climate.copernicus.eu/how-to-api). Store your credentials in `~/.cdsapirc`:
> ```
> $ more ~/.cdsapirc
> url: https://cds.climate.copernicus.eu/api
> key: xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx
> ```

## 7. Download ERA5 from Copernicus CDS

In [ ]:
import cdsapi
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path

ERA5_MSL_FILE  = Path("era5_mocha_msl_1h.nc")
ERA5_WIND_FILE = Path("era5_mocha_wind10m_1h.nc")

# Bay of Bengal bounding box for CDS API [N, W, S, E]
AREA  = [24, 82, 8, 97]
DATES = [f"2023-05-{d:02d}" for d in range(12, 16)]   # 12–15 May 2023
HOURS = [f"{h:02d}:00" for h in range(24)]             # hourly

c = cdsapi.Client()

if not ERA5_MSL_FILE.exists():
    print("Downloading ERA5 mean sea-level pressure …")
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable":     ["mean_sea_level_pressure"],
            "date":         DATES,
            "time":         HOURS,
            "area":         AREA,
            "format":       "netcdf",
        },
        str(ERA5_MSL_FILE),
    )
else:
    print(f"Using cached {ERA5_MSL_FILE}")

if not ERA5_WIND_FILE.exists():
    print("Downloading ERA5 10 m wind components …")
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable":     ["10m_u_component_of_wind", "10m_v_component_of_wind"],
            "date":         DATES,
            "time":         HOURS,
            "area":         AREA,
            "format":       "netcdf",
        },
        str(ERA5_WIND_FILE),
    )
else:
    print(f"Using cached {ERA5_WIND_FILE}")

ds_msl  = xr.open_dataset(ERA5_MSL_FILE)
ds_wind = xr.open_dataset(ERA5_WIND_FILE)
print(ds_msl)

## 8. Derive ERA5 Track and Plot

In [ ]:
import holoviews as hv
import hvplot.pandas
from holoviews import opts

# detect variable / dimension names (ERA5 NetCDF names vary slightly by API version)
msl_var  = next(v for v in ds_msl.data_vars  if "msl" in v.lower() or "pressure" in v.lower())
u_var    = next(v for v in ds_wind.data_vars if v.lower() in ("u10", "u"))
v_var    = next(v for v in ds_wind.data_vars if v.lower() in ("v10", "v"))
lat_dim  = "latitude"   if "latitude"   in ds_msl.dims else "lat"
lon_dim  = "longitude"  if "longitude"  in ds_msl.dims else "lon"
time_dim = "valid_time" if "valid_time" in ds_msl.dims else "time"

# --- derive cyclone centre per timestep = grid point of minimum MSL ---
records = []
for t in ds_msl[time_dim].values:
    msl_t      = ds_msl[msl_var].sel({time_dim: t})
    min_msl_pa = float(msl_t.min())
    if min_msl_pa > 98_500:   # Pa — skip timesteps before the storm organises (~985 hPa)
        continue
    lat_idx, lon_idx = np.unravel_index(int(msl_t.values.argmin()), msl_t.shape)
    lat = float(msl_t[lat_dim][lat_idx])
    lon = float(msl_t[lon_dim][lon_idx])
    u   = float(ds_wind[u_var].sel({time_dim: t, lat_dim: lat, lon_dim: lon}, method="nearest"))
    v   = float(ds_wind[v_var].sel({time_dim: t, lat_dim: lat, lon_dim: lon}, method="nearest"))
    ws  = float(np.sqrt(u**2 + v**2))
    records.append({
        "time":    pd.Timestamp(t),
        "lat":     lat,
        "lon":     lon,
        "msl_hPa": round(min_msl_pa / 100, 1),
        "wind_ms": round(ws, 1),
        "wind_kt": round(ws / 0.514444, 1),
    })

track = pd.DataFrame(records).sort_values("time").reset_index(drop=True)
print(f"Track points : {len(track)}")
print(f"Period       : {track['time'].iloc[0]}  →  {track['time'].iloc[-1]}")
print(f"Min MSL      : {track['msl_hPa'].min():.1f} hPa")
print(f"Max wind     : {track['wind_ms'].max():.1f} m/s  ({track['wind_kt'].max():.0f} kt)")
track.head(10)

In [ ]:
import panel as pn
pn.extension()

msl_hpa = ds_msl[msl_var] / 100  # Pa → hPa
times = ds_msl[time_dim].values

time_slider = pn.widgets.DiscreteSlider(
    name="Time (UTC)",
    options={str(pd.Timestamp(t)): t for t in times},
    width=700,
)

@pn.depends(time_slider)
def cyclone_plot(t):
    t_ts = pd.Timestamp(t)

    pressure = msl_hpa.sel({time_dim: t}).hvplot.contourf(
        x=lon_dim, y=lat_dim,
        geo=True,
        levels=20,
        cmap="RdBu_r",
        clim=(960, 1015),
        alpha=0.6,
        colorbar=True,
        clabel="MSL pressure (hPa)",
        tiles="EsriImagery",
    )

    # all track points up to current time
    track_so_far = track[track["time"] <= t_ts]

    layers = pressure
    if not track_so_far.empty:
        layers = layers * track_so_far.hvplot.points(
            x="lon", y="lat",
            geo=True,
            c="wind_kt",
            cmap="YlOrRd",
            clim=(20, 120),
            size=8,
            colorbar=True,
            clabel="10 m wind (kt)",
            hover_cols=["time", "msl_hPa", "wind_kt"],
        )

    return layers.opts(
        opts.Points(tools=["hover"]),
        opts.Overlay(
            width=750, height=600,
            title=f"Cyclone Mocha — {t_ts.strftime('%Y-%m-%d %H:%M UTC')}",
        ),
    )

pn.Column(time_slider, cyclone_plot)